# Reusable Template: Survival Analysis Project

Copy this notebook as the starting point for a new survival/time-to-event project. Edit only the
**CONFIG** cell and the **Load your data** cell — everything below is written to work generically as
long as your data has a duration column, a 0/1 event column, and optional covariate/group columns.

**Requirements:** `pip install lifelines==0.27.4 statsmodels pandas matplotlib numpy --break-system-packages`


## 1. Configuration — edit this cell for your project

In [ ]:
# ---- EDIT ME ----
DATA_PATH = 'your_data.csv'          # path to a CSV, or None if loading from a package/API instead
DURATION_COL = 'time'                # name of the time-to-event / duration column
EVENT_COL = 'event'                  # name of the 0/1 event indicator (1 = event observed, 0 = censored)
GROUP_COL = None                     # optional: categorical column to compare groups on, e.g. 'treatment'
COVARIATES = []                      # list of column names to use in the Cox PH model, e.g. ['age', 'sex']
TRAIN_FRACTION = 0.8                 # fraction of rows used for training the Cox model; rest is holdout
RANDOM_SEED = 42
# ------------------


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from lifelines import KaplanMeierFitter, ExponentialFitter, WeibullFitter, LogNormalFitter, CoxPHFitter
from lifelines.statistics import logrank_test, multivariate_logrank_test
from lifelines.utils import concordance_index

plt.rcParams['figure.figsize'] = (8, 5)
np.random.seed(RANDOM_SEED)


## 2. Load your data
Replace this cell's body with however your project sources data (CSV, database query, API, `lifelines.datasets`, `statsmodels` R datasets, etc.). The rest of the notebook only assumes a DataFrame called `df` with the columns configured above.

In [ ]:
# Example: load from CSV
# df = pd.read_csv(DATA_PATH)

# Example: load a lifelines built-in dataset instead (comment out the line above and use this)
# from lifelines import datasets
# df = datasets.load_larynx()

df = pd.DataFrame()   # <-- replace with your real data load
assert DURATION_COL in df.columns, f'{DURATION_COL} not found in df'
assert EVENT_COL in df.columns, f'{EVENT_COL} not found in df'
df.head()


## 3. Sanity checks
Run this before modeling anything. Bad event coding or extreme censoring rates silently break every model downstream.

In [ ]:
print('Rows:', len(df))
print('Event rate:', df[EVENT_COL].mean())
print('Duration range:', df[DURATION_COL].min(), '-', df[DURATION_COL].max())
print('Missing values in modeling columns:')
print(df[[DURATION_COL, EVENT_COL] + COVARIATES].isna().sum())

fig, ax = plt.subplots()
ax.hist(df[DURATION_COL], bins=20)
ax.set_title('Distribution of duration'); ax.set_xlabel(DURATION_COL);


## 4. Kaplan-Meier — overall survival curve

In [ ]:
kmf = KaplanMeierFitter()
kmf.fit(df[DURATION_COL], df[EVENT_COL], label='overall')

fig, ax = plt.subplots()
kmf.plot_survival_function(ax=ax)
ax.set_title('Kaplan-Meier — overall'); ax.set_xlabel(DURATION_COL); ax.set_ylabel('Survival probability');

print('Median survival:', kmf.median_survival_time_)


## 5. Kaplan-Meier by group (only runs if `GROUP_COL` is set)

In [ ]:
def km_by_group(df, duration_col, event_col, group_col):
    """Fit and plot one Kaplan-Meier curve per level of group_col; run a log-rank test."""
    fig, ax = plt.subplots()
    fitters = {}
    for level, sub in df.groupby(group_col):
        f = KaplanMeierFitter().fit(sub[duration_col], sub[event_col], label=str(level))
        fitters[level] = f
        f.plot(ax=ax)
    ax.set_title(f'Kaplan-Meier by {group_col}')

    levels = df[group_col].unique()
    if len(levels) == 2:
        a, b = levels
        res = logrank_test(df.loc[df[group_col]==a, duration_col], df.loc[df[group_col]==b, duration_col],
                            df.loc[df[group_col]==a, event_col], df.loc[df[group_col]==b, event_col])
        res.print_summary()
    else:
        res = multivariate_logrank_test(df[duration_col], df[group_col], df[event_col])
        res.print_summary()
    return fitters, res

if GROUP_COL is not None:
    fitters, logrank_result = km_by_group(df, DURATION_COL, EVENT_COL, GROUP_COL)
else:
    print('GROUP_COL not set — skipping group comparison. Set it in the CONFIG cell to enable this section.')


## 6. Parametric model comparison (Exponential / Weibull / LogNormal)

In [ ]:
def compare_parametric_models(durations, events):
    """Fit Exponential/Weibull/LogNormal, return AIC table sorted best-to-worst, plot all + KM."""
    candidates = {'Exponential': ExponentialFitter(), 'Weibull': WeibullFitter(), 'LogNormal': LogNormalFitter()}
    aic = {}
    fig, ax = plt.subplots()
    KaplanMeierFitter().fit(durations, events, label='Kaplan-Meier').plot(ax=ax, ci_show=False, linestyle='--', color='black')
    for name, fitter in candidates.items():
        fitter.fit(durations, events, label=name)
        aic[name] = fitter.AIC_
        fitter.plot_survival_function(ax=ax)
    ax.set_title('Parametric fits vs. Kaplan-Meier (dashed)')
    return pd.Series(aic).sort_values()

compare_parametric_models(df[DURATION_COL], df[EVENT_COL])


## 7. Cox Proportional Hazards (only runs if `COVARIATES` is set)

In [ ]:
def fit_cox(df, duration_col, event_col, covariates, train_fraction, seed):
    """Train/test split, fit CoxPHFitter, print summary, check PH assumption, return the fitted model."""
    model_df = df[[duration_col, event_col] + covariates].dropna()
    train = model_df.sample(frac=train_fraction, random_state=seed)
    test = model_df.drop(train.index)

    cph = CoxPHFitter()
    cph.fit(train, duration_col=duration_col, event_col=event_col)
    cph.print_summary()

    cph.plot()
    plt.title('Cox PH — coefficients within 95% CI')

    print('\nChecking proportional hazards assumption:')
    cph.check_assumptions(train, p_value_threshold=0.05, show_plots=True)

    c_index = concordance_index(train[duration_col], -cph.predict_partial_hazard(train), train[event_col])
    print(f'\nConcordance index (train): {c_index:.3f}')

    return cph, train, test

if COVARIATES:
    cph, train, test = fit_cox(df, DURATION_COL, EVENT_COL, COVARIATES, TRAIN_FRACTION, RANDOM_SEED)
else:
    print('COVARIATES not set — skipping Cox PH. Set it in the CONFIG cell to enable this section.')


## 8. Predict on new / holdout data (only runs if Cox model was fit above)

In [ ]:
if COVARIATES:
    cph.predict_survival_function(test[COVARIATES]).plot(legend=False)
    plt.xlabel(DURATION_COL); plt.ylabel('Survival probability'); plt.title('Predicted survival — holdout set')

    holdout_c_index = concordance_index(test[DURATION_COL], -cph.predict_partial_hazard(test), test[EVENT_COL])
    print(f'Concordance index (holdout): {holdout_c_index:.3f}')


## 9. Notes for future you

- Re-run Section 3 (sanity checks) every time you plug in new data — silent event-coding mistakes are
  the most common source of wrong conclusions in survival analysis.
- If the PH-assumption check in Section 7 fails for a covariate, consider `cph.fit(..., strata=[col])`
  or a time-varying-covariate model (`CoxTimeVaryingFitter`) before trusting the hazard ratio.
- Concordance around 0.5 means the model isn't discriminating between fast/slow failures — more/better
  covariates are needed, not more data rows.
- See `00_Background_Theory.md` for the underlying math and the limitations of each model family.
